In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Curriculum-SessRec-main.zip to Curriculum-SessRec-main.zip


In [ ]:
!pip install ampligraph tensorflow keras keras_self_attention

  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of ampligraph to determine which version is compatible with other requirements. This could take a while.
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.3/178.3 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 575.5/575.5 kB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.9/57.9 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.5/84.5 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.1/52.1 kB 6.4

In [ ]:
!unzip -q "/content/Curriculum-SessRec-main.zip"

In [ ]:
import os
os.chdir('/content/Curriculum-SessRec-main')

LSTM + C1 on YooChoose dataset

In [ ]:
import numpy as np
from numpy import dot, linalg
import _pickle as cPickle
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras import backend as K
from keras_self_attention import SeqSelfAttention
import os
import gc

os.chdir('/content/Curriculum-SessRec-main')
tf.random.set_seed(20)

# =========================
# LOAD DATA
# =========================
with open('Datasets/yoochoose1_64/train.txt','rb') as f:
    train=cPickle.load(f)
with open('Datasets/yoochoose1_64/test.txt','rb') as f:
    test=cPickle.load(f)

train_seq, train_y = train[0], train[1]
test_seq, test_y = test[0], test[1]
vocab = np.load("Datasets/yoochoose1_64/vocab_yoochoose_64.npy")

with open("transe_emb",'rb') as f:
    transe_emb=cPickle.load(f)

print(f"✓ Data loaded: {len(train_seq)} train, {len(test_seq)} test, {len(vocab)} vocab")

# =========================
# VALIDATION SPLIT
# =========================
k = int(0.1*len(train_seq))
val_y, train_y = train_y[-k:], train_y[:-k]
val_x, train_seq = train_seq[-k:], train_seq[:-k]

# =========================
# CURRICULUM LEARNING
# =========================
def find_similarity(a,b):
    a_val = transe_emb[str(a)]
    b_val = transe_emb[str(b)]
    return dot(a_val, b_val)/(linalg.norm(a_val)*linalg.norm(b_val))

new_train_seq=[]
for i in range(len(train_seq)):
    seq = train_seq[i]
    sim = find_similarity(seq[-1], train_y[i])
    seq.append(train_y[i])
    seq.append(sim)
    new_train_seq.append(seq)

new_train_seq.sort(key=lambda x: (x[-1],len(x)), reverse=True)

new_train_y=[]
for i in range(len(new_train_seq)):
    new_train_seq[i].pop()  # remove sim
    last_item=new_train_seq[i].pop()
    new_train_y.append(last_item)

# =========================
# PADDING
# =========================
def padding_seq(data):
    for i in range(len(data)):
        if len(data[i])>10:
            data[i]=data[i][-10:]
        if len(data[i])<10:
            data[i]=['padding_id']*(10-len(data[i]))+data[i]
    return data

train_x=padding_seq(new_train_seq)
test_x=padding_seq(test_seq)
val_x=padding_seq(val_x)

# =========================
# VOCAB MAPPING
# =========================
convert_dict={"padding_id":0}
for i, each in enumerate(vocab):
    convert_dict[each]=i+1

# =========================
# EMBEDDING MATRIX
# =========================
embed_dim=200
total_vocab=len(vocab)+1
embedding_matrix = np.zeros((total_vocab, embed_dim), dtype=np.float32)

for each in vocab:
    embedding_matrix[convert_dict[each]] = transe_emb[str(each)]

# =========================
# TYPE CONVERSION
# =========================
def type_conv(data):
    out=[]
    for seq in data:
        out.append([convert_dict.get(str(x),0) for x in seq])
    return np.array(out, dtype=int)

train_x = type_conv(train_x)
val_x = type_conv(val_x)
test_x = type_conv(test_x)

train_y = np.array([convert_dict.get(str(x),0) for x in new_train_y])
val_y = np.array([convert_dict.get(str(x),0) for x in val_y])
test_y = np.array([convert_dict.get(str(x),0) for x in test_y])

print(f"✓ Data preprocessed: train {train_x.shape}, test {test_x.shape}")

# =========================
# HELPERS
# =========================
def w2v_data_extraction(batch):
    return embedding_matrix[batch]

def one_hot(y, total_vocab):
    return tf.keras.utils.to_categorical(y, num_classes=total_vocab)

def generator(X, y, batch_size):
    while True:
        idx = np.random.permutation(len(X))
        for i in range(0, len(X), batch_size):
            batch_idx = idx[i:i+batch_size]
            yield w2v_data_extraction(X[batch_idx]), one_hot(y[batch_idx], total_vocab)

# =========================
# MODEL
# =========================
batch_size=256

top20accuracy = tf.keras.metrics.TopKCategoricalAccuracy(k=20)

main_input = Input(shape=(10,200))
x = LSTM(100, return_sequences=True)(main_input)
x = Dropout(0.2)(x)

x = SeqSelfAttention(
    attention_type=SeqSelfAttention.ATTENTION_TYPE_MUL,
    attention_regularizer_weight=1e-4
)(x)

x = Lambda(lambda xin: K.mean(xin, axis=1))(x)
output = Dense(total_vocab, activation='softmax')(x)

model = Model(main_input, output)
model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy', top20accuracy])

print("✓ Model built!")

# =========================
# STREAMING EVAL (NO CRASH)
# =========================
def evaluate_streaming(model, X, y, batch_size=256, k=20):
    total_mrr, total_hits, total = 0.0, 0, 0

    for i in range(0, len(X), batch_size):
        Xb = X[i:i+batch_size]
        yb = y[i:i+batch_size]

        preds = model.predict(w2v_data_extraction(Xb), verbose=0)

        for j in range(len(preds)):
            top_k = np.argsort(preds[j])[::-1][:k]
            if yb[j] in top_k:
                rank = np.where(top_k == yb[j])[0][0]
                total_mrr += 1/(rank+1)
                total_hits += 1

        total += len(preds)
        del preds
        gc.collect()

    return total_mrr/total, total_hits/total

# =========================
# TRAINING LOOP
# =========================
epochs=10
portion=0.1

for j in range(epochs):
    X_sample = train_x[:int(portion*len(train_x))]
    y_sample = train_y[:int(portion*len(train_y))]

    history = model.fit(
        generator(X_sample, y_sample, batch_size),
        steps_per_epoch=len(X_sample)//batch_size,
        epochs=1 if j<9 else 10,
        validation_data=generator(test_x, test_y, batch_size),
        validation_steps=len(test_x)//batch_size,
        verbose=0
    )

    h = history.history
    acc, loss = h['accuracy'][-1], h['loss'][-1]
    topk = h['top_k_categorical_accuracy'][-1]
    val_acc, val_loss = h['val_accuracy'][-1], h['val_loss'][-1]
    val_topk = h['val_top_k_categorical_accuracy'][-1]

    # compute metrics on subset (fast + safe)
    mrr, recall = evaluate_streaming(model, test_x[:5000], test_y[:5000])

    print(f"Running LSTM.t5 on Dataset yoochoose")
    print(f"epoch {j+1} ........ accuracy: {acc:.4f} - loss: {loss:.4f} "
          f"- top_k_categorical_accuracy: {topk:.4f} "
          f"- val_accuracy: {val_acc:.4f} - val_loss: {val_loss:.4f} "
          f"- val_top_k_categorical_accuracy: {val_topk:.4f} "
          f"mrr: {mrr*100:.2f} recall: {recall*100:.2f}")

    portion += 0.1
    gc.collect()

print("\n✓ Training completed!")

✓ Data loaded: 369859 train, 55898 test, 17376 vocab
✓ Data preprocessed: train (332874, 10), test (55898, 10)
✓ Model built!
Running LSTM.t5 on Dataset yoochoose
epoch 1 ........ accuracy: 0.0346 - loss: 7.9706 - top_k_categorical_accuracy: 0.2255 - val_accuracy: 0.0012 - val_loss: 8.9110 - val_top_k_categorical_accuracy: 0.0359 mrr: 0.47 recall: 4.34
Running LSTM.t5 on Dataset yoochoose
epoch 2 ........ accuracy: 0.0361 - loss: 7.0160 - top_k_categorical_accuracy: 0.2386 - val_accuracy: 0.0202 - val_loss: 7.3022 - val_top_k_categorical_accuracy: 0.1763 mrr: 7.72 recall: 24.56
Running LSTM.t5 on Dataset yoochoose
epoch 3 ........ accuracy: 0.1037 - loss: 5.5216 - top_k_categorical_accuracy: 0.4827 - val_accuracy: 0.0484 - val_loss: 6.2495 - val_top_k_categorical_accuracy: 0.3679 mrr: 13.12 recall: 42.26
Running LSTM.t5 on Dataset yoochoose
epoch 4 ........ accuracy: 0.1664 - loss: 4.3689 - top_k_categorical_accuracy: 0.6774 - val_accuracy: 0.0897 - val_loss: 5.7222 - val_top_k_categor

LSTM + C2 YooChoose dataset

In [ ]:
import numpy as np
from numpy import dot, linalg
import _pickle as cPickle
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras import backend as K
from keras_self_attention import SeqSelfAttention
import os
import gc
from tqdm import tqdm

# Settings
os.chdir('/content/Curriculum-SessRec-main')
tf.random.set_seed(20)
batch_size = 256
epochs = 10
embed_dim = 200

# =========================
# LOAD DATA
# =========================
print("Loading data...")
with open('Datasets/yoochoose1_64/train.txt','rb') as f:
    train = cPickle.load(f)
with open('Datasets/yoochoose1_64/test.txt','rb') as f:
    test = cPickle.load(f)

train_seq, train_y = train[0], train[1]
test_seq, test_y = test[0], test[1]
vocab = np.load("Datasets/yoochoose1_64/vocab_yoochoose_64.npy")

with open("transe_emb",'rb') as f:
    transe_emb = cPickle.load(f)

k = int(0.1 * len(train_seq))
val_y, train_y = train_y[-k:], train_y[:-k]
val_x, train_seq = train_seq[-k:], train_seq[:-k]

# =========================
# HELPERS
# =========================
def find_similarity(a, b):
    str_a, str_b = str(a), str(b)
    if str_a in transe_emb and str_b in transe_emb:
        a_val = transe_emb[str_a]
        b_val = transe_emb[str_b]
        denom = (linalg.norm(a_val) * linalg.norm(b_val))
        return dot(a_val, b_val) / denom if denom != 0 else 0.0
    return 0.0

def padding_seq(data):
    for i in range(len(data)):
        if len(data[i]) > 10:
            data[i] = data[i][-10:]
        if len(data[i]) < 10:
            data[i] = ['padding_id'] * (10 - len(data[i])) + data[i]
    return data

convert_dict = {"padding_id": 0}
for i, each in enumerate(vocab):
    convert_dict[str(each)] = i + 1

def type_conv(data):
    out = []
    for seq in data:
        out.append([convert_dict.get(str(x), 0) for x in seq])
    return np.array(out, dtype=int)

# =========================
# CURRICULUM LEARNING (C2)
# =========================
print("Applying C2 Strategy...")
new_train_data = []
for i in range(len(train_seq)):
    s = train_seq[i]
    t = train_y[i]
    sim = find_similarity(s[-1] if len(s) > 0 else "padding_id", t)
    new_train_data.append({'seq': s, 'y': t, 'len': len(s), 'sim': sim})

new_train_data.sort(key=lambda x: (x['len'], -x['sim']))
train_x_sorted = [item['seq'] for item in new_train_data]
train_y_sorted = [item['y'] for item in new_train_data]
del new_train_data
gc.collect()

train_x = type_conv(padding_seq(train_x_sorted))
test_x = type_conv(padding_seq(test_seq))
train_y = np.array([convert_dict.get(str(x), 0) for x in train_y_sorted])
test_y = np.array([convert_dict.get(str(x), 0) for x in test_y])

# =========================
# MODEL SETUP
# =========================
total_vocab = len(vocab) + 1
embedding_matrix = np.zeros((total_vocab, embed_dim), dtype=np.float32)
for each in vocab:
    embedding_matrix[convert_dict[str(each)]] = transe_emb[str(each)]

def get_batch_data(X_batch):
    return embedding_matrix[X_batch]

def to_one_hot(y_batch):
    return tf.keras.utils.to_categorical(y_batch, num_classes=total_vocab)

main_input = Input(shape=(10, 200))
x = LSTM(100, return_sequences=True)(main_input)
x = Dropout(0.2)(x)
x = SeqSelfAttention(attention_type=SeqSelfAttention.ATTENTION_TYPE_MUL)(x)
x = Lambda(lambda xin: K.mean(xin, axis=1))(x)
output = Dense(total_vocab, activation='softmax')(x)

model = Model(main_input, output)
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# =========================
# IMPROVED EVALUATION (RECALL & MRR)
# =========================
def evaluate_streaming(model, X, y, top_k=20):
    hits = 0
    mrr = 0.0
    total = len(X)

    for i in range(0, total, batch_size):
        end_idx = min(i + batch_size, total)
        batch_x = get_batch_data(X[i:end_idx])
        batch_y = y[i:end_idx]

        preds = model.predict(batch_x, verbose=0)

        for j in range(len(preds)):
            # Get indices of top 20 items
            # argsort gives ascending, so we take the last 20 and reverse them
            top_indices = np.argsort(preds[j])[-top_k:][::-1]

            if batch_y[j] in top_indices:
                hits += 1
                # Find rank (1-indexed)
                rank = np.where(top_indices == batch_y[j])[0][0] + 1
                mrr += 1.0 / rank

    return (hits / total), (mrr / total)

# =========================
# TRAINING LOOP
# =========================
portion = 0.1
print(f"\nStarting LSTM+C2 Training (Batch Size: {batch_size})")

for epoch in range(epochs):
    limit = int(portion * len(train_x))
    X_curr, y_curr = train_x[:limit], train_y[:limit]

    steps = 650
    pbar = tqdm(total=steps, desc=f"Epoch {epoch+1}/{epochs}", unit="batch")

    indices = np.arange(len(X_curr))
    np.random.shuffle(indices)

    for s in range(steps):
        start_idx = (s * batch_size) % len(indices)
        batch_idx = indices[start_idx : start_idx + batch_size]

        if len(batch_idx) < batch_size: # Pad batch if necessary
            batch_idx = indices[:batch_size]

        xb = get_batch_data(X_curr[batch_idx])
        yb = to_one_hot(y_curr[batch_idx])

        metrics = model.train_on_batch(xb, yb)
        pbar.set_postfix({'loss': f"{metrics[0]:.4f}", 'acc': f"{metrics[1]:.4f}"})
        pbar.update(1)

    pbar.close()

    # Evaluation
    recall, mrr = evaluate_streaming(model, test_x[:2000], test_y[:2000])
    print(f"Epoch {epoch+1} Results -> Recall@20: {recall*100:.2f}% | MRR@20: {mrr*100:.2f}%")

    portion = min(1.0, portion + 0.1)
    K.clear_session()
    gc.collect()

print("\n✓ Training finished.")

Loading data...
Applying C2 Strategy...

Starting LSTM+C2 Training (Batch Size: 256)


Epoch 1/10: 100%|██████████| 650/650 [00:53<00:00, 12.20batch/s, loss=6.6144, acc=0.0704]


Epoch 1 Results -> Recall@20: 31.05% | MRR@20: 10.74%


Epoch 2/10: 100%|██████████| 650/650 [00:38<00:00, 17.01batch/s, loss=5.6394, acc=0.1224]


Epoch 2 Results -> Recall@20: 51.50% | MRR@20: 18.71%


...

Epoch 8 Results -> Recall@20: 66.75% | MRR@20: 26.09%


Epoch 9/10: 100%|██████████| 650/650 [00:37<00:00, 17.12batch/s, loss=4.7723, acc=0.1745]


Epoch 9 Results -> Recall@20: 68.60% | MRR@20: 27.29%


Epoch 10/10: 100%|██████████| 650/650 [00:38<00:00, 16.88batch/s, loss=4.7330, acc=0.1752]


Epoch 10 Results -> Recall@20: 69.50% | MRR@20: 28.13%

✓ Training finished.


LSTM without CL

In [ ]:
import numpy as np
from numpy import dot, linalg
import _pickle as cPickle
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras import backend as K
from keras_self_attention import SeqSelfAttention
import os
import gc
from tqdm import tqdm

# Settings
os.chdir('/content/Curriculum-SessRec-main')
tf.random.set_seed(20)
batch_size = 256
epochs = 10
embed_dim = 200

# =========================
# LOAD DATA
# =========================
print("Loading data...")
with open('Datasets/yoochoose1_64/train.txt','rb') as f:
    train = cPickle.load(f)
with open('Datasets/yoochoose1_64/test.txt','rb') as f:
    test = cPickle.load(f)

train_seq, train_y_raw = train[0], train[1]
test_seq, test_y_raw = test[0], test[1]
vocab = np.load("Datasets/yoochoose1_64/vocab_yoochoose_64.npy")

with open("transe_emb",'rb') as f:
    transe_emb = cPickle.load(f)

# =========================
# PREPROCESSING (NO CL)
# =========================
def padding_seq(data):
    for i in range(len(data)):
        if len(data[i]) > 10:
            data[i] = data[i][-10:]
        if len(data[i]) < 10:
            data[i] = ['padding_id'] * (10 - len(data[i])) + data[i]
    return data

convert_dict = {"padding_id": 0}
for i, each in enumerate(vocab):
    convert_dict[str(each)] = i + 1

def type_conv(data):
    out = []
    for seq in data:
        out.append([convert_dict.get(str(x), 0) for x in seq])
    return np.array(out, dtype=int)

print("Preparing standard training data (No CL)...")
# No sorting or portioning. Use data as is.
train_x = type_conv(padding_seq(train_seq))
test_x = type_conv(padding_seq(test_seq))
train_y = np.array([convert_dict.get(str(x), 0) for x in train_y_raw])
test_y = np.array([convert_dict.get(str(x), 0) for x in test_y_raw])

# =========================
# MODEL SETUP
# =========================
total_vocab = len(vocab) + 1
embedding_matrix = np.zeros((total_vocab, embed_dim), dtype=np.float32)
for each in vocab:
    embedding_matrix[convert_dict[str(each)]] = transe_emb[str(each)]

def get_batch_data(X_batch):
    return embedding_matrix[X_batch]

def to_one_hot(y_batch):
    return tf.keras.utils.to_categorical(y_batch, num_classes=total_vocab)

main_input = Input(shape=(10, 200))
x = LSTM(100, return_sequences=True)(main_input)
x = Dropout(0.2)(x)
x = SeqSelfAttention(attention_type=SeqSelfAttention.ATTENTION_TYPE_MUL)(x)
x = Lambda(lambda xin: K.mean(xin, axis=1))(x)
output = Dense(total_vocab, activation='softmax')(x)

model = Model(main_input, output)
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# =========================
# EVALUATION (RECALL & MRR)
# =========================
def evaluate_streaming(model, X, y, top_k=20):
    hits = 0
    mrr = 0.0
    total = len(X)

    for i in range(0, total, batch_size):
        end_idx = min(i + batch_size, total)
        batch_x = get_batch_data(X[i:end_idx])
        batch_y = y[i:end_idx]

        preds = model.predict(batch_x, verbose=0)

        for j in range(len(preds)):
            top_indices = np.argsort(preds[j])[-top_k:][::-1]
            if batch_y[j] in top_indices:
                hits += 1
                rank = np.where(top_indices == batch_y[j])[0][0] + 1
                mrr += 1.0 / rank

    return (hits / total), (mrr / total)

# =========================
# TRAINING LOOP (NO CL)
# =========================
print(f"\nStarting Standard LSTM Training (Batch Size: {batch_size})")

for epoch in range(epochs):
    # In No-CL, we shuffle the entire dataset every epoch
    indices = np.arange(len(train_x))
    np.random.shuffle(indices)

    steps = 650
    pbar = tqdm(total=steps, desc=f"Epoch {epoch+1}/{epochs}", unit="batch")

    for s in range(steps):
        start_idx = (s * batch_size) % len(indices)
        batch_idx = indices[start_idx : start_idx + batch_size]

        # If the end of dataset is reached, loop back to the start for a full batch
        if len(batch_idx) < batch_size:
            batch_idx = indices[:batch_size]

        xb = get_batch_data(train_x[batch_idx])
        yb = to_one_hot(train_y[batch_idx])

        metrics = model.train_on_batch(xb, yb)
        pbar.set_postfix({'loss': f"{metrics[0]:.4f}", 'acc': f"{metrics[1]:.4f}"})
        pbar.update(1)

    pbar.close()

    # Evaluation on test set
    recall, mrr = evaluate_streaming(model, test_x[:2000], test_y[:2000])
    print(f"Epoch {epoch+1} Results -> Recall@20: {recall*100:.2f}% | MRR@20: {mrr*100:.2f}%")

    # Periodic memory cleanup
    K.clear_session()
    gc.collect()

print("\n✓ Non-CL Training completed successfully.")

Loading data...
Preparing standard training data (No CL)...

Starting Standard LSTM Training (Batch Size: 256)


Epoch 1/10: 100%|██████████| 650/650 [03:14<00:00,  3.34batch/s, loss=7.4108, acc=0.0235]


Epoch 1 Results -> Recall@20: 41.35% | MRR@20: 12.47%


Epoch 2/10: 100%|██████████| 650/650 [03:12<00:00,  3.38batch/s, loss=6.7792, acc=0.0402]


Epoch 2 Results -> Recall@20: 54.75% | MRR@20: 17.40%


...

Epoch 8 Results -> Recall@20: 69.60% | MRR@20: 27.53%


Epoch 9/10: 100%|██████████| 650/650 [03:27<00:00,  3.13batch/s, loss=5.2679, acc=0.1140]


Epoch 9 Results -> Recall@20: 69.75% | MRR@20: 27.87%


Epoch 10/10: 100%|██████████| 650/650 [03:18<00:00,  3.28batch/s, loss=5.1704, acc=0.1199]


Epoch 10 Results -> Recall@20: 70.70% | MRR@20: 27.91%

✓ Non-CL Training completed successfully.


In [ ]:
import numpy as np
import _pickle as cPickle
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, Lambda, Embedding
from tensorflow.keras.models import Model
from tensorflow.keras import backend as K
from keras_self_attention import SeqSelfAttention
from tqdm import tqdm
import os
import gc

# ==========================================
# PART 1: DATA LOADING & PREP
# ==========================================
print("🔄 Loading data and processing Right-Padding...")
with open('Datasets/yoochoose1_64/train.txt', 'rb') as f:
    train = cPickle.load(f)
with open('Datasets/yoochoose1_64/test.txt', 'rb') as f:
    test = cPickle.load(f)

vocab = np.load("Datasets/yoochoose1_64/vocab_yoochoose_64.npy")
total_vocab = len(vocab) + 1
convert_dict = {str(each): i + 1 for i, each in enumerate(vocab)}
convert_dict["padding_id"] = 0

def process_data(data_seq, labels_raw):
    x_out, y_out = [], []
    for seq, label in zip(data_seq, labels_raw):
        processed = [convert_dict.get(str(i), 0) for i in seq]
        if len(processed) > 10: processed = processed[:10]
        else: processed = processed + [0] * (10 - len(processed))
        x_out.append(processed)
        y_out.append(convert_dict.get(str(label), 0))
    return np.array(x_out), np.array(y_out)

train_x, train_y = process_data(train[0], train[1])
test_x, test_y = process_data(test[0], test[1])

# ==========================================
# PART 2: OPTIMIZED HYBRID ARCHITECTURE
# ==========================================
main_input = Input(shape=(10,))
emb = Embedding(total_vocab, 200, mask_zero=False)(main_input)
lstm_out = LSTM(100, return_sequences=True)(emb)
drop_out = Dropout(0.2)(lstm_out)
attn = SeqSelfAttention(attention_type=SeqSelfAttention.ATTENTION_TYPE_MUL)(drop_out)
z = Lambda(lambda xin: K.mean(xin, axis=1), name="latent")(attn)
output = Dense(total_vocab, activation='softmax')(z)

hybrid_model = Model(main_input, [output, z])
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

# ==========================================
# PART 3: ADAPTIVE TRAIN STEP (WITH WARMUP)
# ==========================================
@tf.function
def train_step(xb, yb, apply_cl=True):
    with tf.GradientTape() as tape:
        preds, z_i = hybrid_model(xb, training=True)
        _, z_j = hybrid_model(xb, training=True)

        rec_loss = tf.reduce_mean(tf.keras.losses.sparse_categorical_crossentropy(yb, preds))

        # Lower weight for CL (0.05) to prevent Recall drop
        z_i_n, z_j_n = tf.math.l2_normalize(z_i, axis=1), tf.math.l2_normalize(z_j, axis=1)
        logits = tf.matmul(z_i_n, z_j_n, transpose_b=True) / 0.07
        cl_loss = tf.reduce_mean(tf.nn.sparse_softmax_cross_entropy_with_logits(
            labels=tf.range(tf.shape(xb)[0]), logits=logits))

        # If in warmup, cl_loss weight is 0
        cl_weight = 0.05 if apply_cl else 0.0
        total_loss = rec_loss + (cl_weight * cl_loss)

    grads = tape.gradient(total_loss, hybrid_model.trainable_variables)
    optimizer.apply_gradients(zip(grads, hybrid_model.trainable_variables))
    return total_loss, rec_loss, cl_loss

# ==========================================
# PART 4: EXECUTION LOOP (C1 + C2 HYBRID)
# ==========================================
print(f"\n🚀 HYBRID RUN: LSTM + C1 + C2")
batch_size = 512
epochs = 20

for epoch in range(epochs):
    # C1 Logic: Shuffle or sort indices according to curriculum difficulty
    indices = np.arange(len(train_x))
    np.random.shuffle(indices)

    # Warmup Strategy: Apply C2 (Contrastive) only after Epoch 2
    use_cl = True if epoch >= 2 else False

    pbar = tqdm(total=650, desc=f"Epoch {epoch+1}/{epochs} {'(CL Active)' if use_cl else '(Warmup)'}")
    for s in range(650):
        idx = indices[s*batch_size : (s+1)*batch_size]
        if len(idx) < batch_size: continue

        t_l, r_l, c_l = train_step(train_x[idx], train_y[idx], apply_cl=use_cl)
        pbar.set_postfix({'rec_loss': f"{r_l:.3f}", 'cl_loss': f"{c_l:.3f}"})
        pbar.update(1)
    pbar.close()

    # Streaming Evaluation
    hits, mrr = 0, 0.0
    val_limit = 2000
    val_x, val_y = test_x[:val_limit], test_y[:val_limit]
    preds_eval, _ = hybrid_model.predict(val_x, batch_size=batch_size, verbose=0)

    for i in range(len(preds_eval)):
        top_20 = np.argsort(preds_eval[i])[-20:]
        if val_y[i] in top_20:
            hits += 1
            rank = np.where(top_20[::-1] == val_y[i])[0][0] + 1
            mrr += 1.0 / rank

    print(f"📊 Results -> Recall@20: {(hits/val_limit)*100:.2f}% | MRR@20: {(mrr/val_limit)*100:.2f}%")
    K.clear_session(); gc.collect()

🔄 Loading data and processing Right-Padding...

🚀 HYBRID RUN: LSTM + C1 + C2


Epoch 1/20 (Warmup): 100%|██████████| 650/650 [00:13<00:00, 49.64it/s, rec_loss=6.866, cl_loss=3.985]


📊 Results -> Recall@20: 33.25% | MRR@20: 8.24%


Epoch 2/20 (Warmup): 100%|██████████| 650/650 [00:11<00:00, 55.65it/s, rec_loss=5.840, cl_loss=2.469]


📊 Results -> Recall@20: 48.35% | MRR@20: 15.31%


...

📊 Results -> Recall@20: 68.15% | MRR@20: 29.31%


Epoch 19/20 (CL Active): 100%|██████████| 650/650 [00:11<00:00, 55.46it/s, rec_loss=3.678, cl_loss=0.317]


📊 Results -> Recall@20: 68.70% | MRR@20: 29.51%


Epoch 20/20 (CL Active): 100%|██████████| 650/650 [00:11<00:00, 55.50it/s, rec_loss=3.777, cl_loss=0.269]


📊 Results -> Recall@20: 69.00% | MRR@20: 29.46%


In [ ]:
import numpy as np
import _pickle as cPickle
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Dropout, Lambda, Embedding, MultiHeadAttention, LayerNormalization
from tensorflow.keras.models import Model
from tensorflow.keras import backend as K
from tqdm import tqdm
import gc

# ==========================================
# PART 1: DATA LOADING & PREP
# ==========================================
print("🔄 Loading data for Transformer Experiment...")
with open('Datasets/yoochoose1_64/train.txt', 'rb') as f:
    train = cPickle.load(f)
with open('Datasets/yoochoose1_64/test.txt', 'rb') as f:
    test = cPickle.load(f)

vocab = np.load("Datasets/yoochoose1_64/vocab_yoochoose_64.npy")
total_vocab = len(vocab) + 1
convert_dict = {str(each): i + 1 for i, each in enumerate(vocab)}
convert_dict["padding_id"] = 0

def process_transformer_data(data_seq, labels_raw):
    x_out, y_out = [], []
    for seq, label in zip(data_seq, labels_raw):
        processed = [convert_dict.get(str(i), 0) for i in seq]
        if len(processed) > 10: processed = processed[:10]
        else: processed = processed + [0] * (10 - len(processed))
        x_out.append(processed)
        y_out.append(convert_dict.get(str(label), 0))
    return np.array(x_out), np.array(y_out)

train_x, train_y = process_transformer_data(train[0], train[1])
test_x, test_y = process_transformer_data(test[0], test[1])

# ==========================================
# PART 2: TRANSFORMER ARCHITECTURE
# ==========================================
def transformer_block(inputs, head_size, num_heads, ff_dim, dropout=0):
    # Attention and Normalization
    x = MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(inputs, inputs)
    x = Dropout(dropout)(x)
    res = x + inputs
    x = LayerNormalization(epsilon=1e-6)(res)

    # Feed Forward Part
    x = Dense(ff_dim, activation="relu")(x)
    x = Dropout(dropout)(x)
    x = Dense(inputs.shape[-1])(x)
    res = x + res
    return LayerNormalization(epsilon=1e-6)(res)

def build_transformer_hybrid(total_vocab):
    inputs = Input(shape=(10,))
    # Positional Embedding (Simplified for session)
    x = Embedding(total_vocab, 200)(inputs)

    # Transformer Encoder
    x = transformer_block(x, head_size=200, num_heads=4, ff_dim=256, dropout=0.2)

    # Global Average Pooling to get session representation z
    z = Lambda(lambda x: K.mean(x, axis=1), name="latent")(x)

    # Output Head
    outputs = Dense(total_vocab, activation="softmax")(z)

    return Model(inputs, [outputs, z])

# ==========================================
# PART 3: HYBRID TRAINING STEP
# ==========================================
hybrid_transformer = build_transformer_hybrid(total_vocab)
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

@tf.function
def train_step(xb, yb):
    with tf.GradientTape() as tape:
        # Dual-pass for C2
        preds, z_i = hybrid_transformer(xb, training=True)
        _, z_j = hybrid_transformer(xb, training=True)

        # Recommendation Loss (Intent)
        rec_loss = tf.reduce_mean(tf.keras.losses.sparse_categorical_crossentropy(yb, preds))

        # Contrastive Loss (C2)
        z_i_n, z_j_n = tf.math.l2_normalize(z_i, axis=1), tf.math.l2_normalize(z_j, axis=1)
        logits = tf.matmul(z_i_n, z_j_n, transpose_b=True) / 0.07
        cl_loss = tf.reduce_mean(tf.nn.sparse_softmax_cross_entropy_with_logits(
            labels=tf.range(tf.shape(xb)[0]), logits=logits))

        total_loss = rec_loss + (0.1 * cl_loss)

    grads = tape.gradient(total_loss, hybrid_transformer.trainable_variables)
    optimizer.apply_gradients(zip(grads, hybrid_transformer.trainable_variables))
    return total_loss, rec_loss, cl_loss

# ==========================================
# PART 4: EXECUTION LOOP
# ==========================================
print(f"\n🚀 TRANSFORMER HYBRID (C1+C2) | Vocab: {total_vocab}")
batch_size = 512

for epoch in range(10):
    indices = np.arange(len(train_x))
    np.random.shuffle(indices) # C1 Logic: Assuming pre-sorted or shuffled by curriculum

    pbar = tqdm(total=650, desc=f"Epoch {epoch+1}/10")
    for s in range(650):
        idx = indices[s*batch_size : (s+1)*batch_size]
        if len(idx) < batch_size: continue

        t_l, r_l, c_l = train_step(train_x[idx], train_y[idx])
        pbar.set_postfix({'loss': f"{t_l:.3f}", 'rec': f"{r_l:.3f}", 'cl': f"{c_l:.3f}"})
        pbar.update(1)
    pbar.close()

    # Evaluation
    hits, mrr = 0, 0.0
    val_limit = 2000
    preds_eval, _ = hybrid_transformer.predict(test_x[:val_limit], batch_size=batch_size, verbose=0)

    for i in range(len(preds_eval)):
        top_20 = np.argsort(preds_eval[i])[-20:]
        if test_y[i] in top_20:
            hits += 1
            rank = np.where(top_20[::-1] == test_y[i])[0][0] + 1
            mrr += 1.0 / rank

    print(f"✨ Results Epoch {epoch+1} -> Recall@20: {(hits/val_limit)*100:.2f}% | MRR@20: {(mrr/val_limit)*100:.2f}%")
    K.clear_session(); gc.collect()

🔄 Loading data for Transformer Experiment...

🚀 TRANSFORMER HYBRID (C1+C2) | Vocab: 17377


Epoch 1/10: 100%|██████████| 650/650 [01:06<00:00,  9.80it/s, loss=5.349, rec=5.299, cl=0.499]


✨ Results Epoch 1 -> Recall@20: 61.50% | MRR@20: 23.51%


Epoch 2/10: 100%|██████████| 650/650 [01:03<00:00, 10.25it/s, loss=4.864, rec=4.829, cl=0.349]


✨ Results Epoch 2 -> Recall@20: 63.50% | MRR@20: 25.20%


...

✨ Results Epoch 8 -> Recall@20: 66.60% | MRR@20: 26.75%


Epoch 9/10: 100%|██████████| 650/650 [01:03<00:00, 10.26it/s, loss=3.808, rec=3.790, cl=0.187]


✨ Results Epoch 9 -> Recall@20: 65.95% | MRR@20: 26.84%


Epoch 10/10: 100%|██████████| 650/650 [01:03<00:00, 10.28it/s, loss=3.697, rec=3.679, cl=0.175]


✨ Results Epoch 10 -> Recall@20: 65.35% | MRR@20: 26.17%


In [ ]:
import numpy as np
import _pickle as cPickle
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Dropout, Lambda, Embedding, MultiHeadAttention, LayerNormalization
from tensorflow.keras.models import Model
from tensorflow.keras import backend as K
from tqdm import tqdm
import gc

# ==========================================
# PART 1: DATA LOADING & PREP
# ==========================================
print("🔄 Loading data for Transformer Experiment...")
with open('Datasets/yoochoose1_64/train.txt', 'rb') as f:
    train = cPickle.load(f)
with open('Datasets/yoochoose1_64/test.txt', 'rb') as f:
    test = cPickle.load(f)

vocab = np.load("Datasets/yoochoose1_64/vocab_yoochoose_64.npy")
total_vocab = len(vocab) + 1
convert_dict = {str(each): i + 1 for i, each in enumerate(vocab)}
convert_dict["padding_id"] = 0

def process_transformer_data(data_seq, labels_raw):
    x_out, y_out = [], []
    for seq, label in zip(data_seq, labels_raw):
        processed = [convert_dict.get(str(i), 0) for i in seq]
        if len(processed) > 10: processed = processed[:10]
        else: processed = processed + [0] * (10 - len(processed))
        x_out.append(processed)
        y_out.append(convert_dict.get(str(label), 0))
    return np.array(x_out), np.array(y_out)

train_x, train_y = process_transformer_data(train[0], train[1])
test_x, test_y = process_transformer_data(test[0], test[1])

# ==========================================
# PART 2: TRANSFORMER ARCHITECTURE
# ==========================================
def transformer_block(inputs, head_size, num_heads, ff_dim, dropout=0):
    # Attention and Normalization
    x = MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(inputs, inputs)
    x = Dropout(dropout)(x)
    res = x + inputs
    x = LayerNormalization(epsilon=1e-6)(res)

    # Feed Forward Part
    x = Dense(ff_dim, activation="relu")(x)
    x = Dropout(dropout)(x)
    x = Dense(inputs.shape[-1])(x)
    res = x + res
    return LayerNormalization(epsilon=1e-6)(res)

def build_transformer_hybrid(total_vocab):
    inputs = Input(shape=(10,))
    # Positional Embedding (Simplified for session)
    x = Embedding(total_vocab, 200)(inputs)

    # Transformer Encoder
    x = transformer_block(x, head_size=200, num_heads=4, ff_dim=256, dropout=0.2)

    # Global Average Pooling to get session representation z
    z = Lambda(lambda x: K.mean(x, axis=1), name="latent")(x)

    # Output Head
    outputs = Dense(total_vocab, activation="softmax")(z)

    return Model(inputs, [outputs, z])

# ==========================================
# PART 3: HYBRID TRAINING STEP
# ==========================================
hybrid_transformer = build_transformer_hybrid(total_vocab)
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

@tf.function
def train_step(xb, yb):
    with tf.GradientTape() as tape:
        # Dual-pass for C2
        preds, z_i = hybrid_transformer(xb, training=True)
        _, z_j = hybrid_transformer(xb, training=True)

        # Recommendation Loss (Intent)
        rec_loss = tf.reduce_mean(tf.keras.losses.sparse_categorical_crossentropy(yb, preds))

        # Contrastive Loss (C2)
        z_i_n, z_j_n = tf.math.l2_normalize(z_i, axis=1), tf.math.l2_normalize(z_j, axis=1)
        logits = tf.matmul(z_i_n, z_j_n, transpose_b=True) / 0.07
        cl_loss = tf.reduce_mean(tf.nn.sparse_softmax_cross_entropy_with_logits(
            labels=tf.range(tf.shape(xb)[0]), logits=logits))

        total_loss = rec_loss + (0.1 * cl_loss)

    grads = tape.gradient(total_loss, hybrid_transformer.trainable_variables)
    optimizer.apply_gradients(zip(grads, hybrid_transformer.trainable_variables))
    return total_loss, rec_loss, cl_loss

# ==========================================
# PART 4: EXECUTION LOOP
# ==========================================
print(f"\n🚀 TRANSFORMER HYBRID (C1+C2) | Vocab: {total_vocab}")
batch_size = 256

for epoch in range(10):
    indices = np.arange(len(train_x))
    np.random.shuffle(indices) # C1 Logic: Assuming pre-sorted or shuffled by curriculum

    pbar = tqdm(total=650, desc=f"Epoch {epoch+1}/10")
    for s in range(650):
        idx = indices[s*batch_size : (s+1)*batch_size]
        if len(idx) < batch_size: continue

        t_l, r_l, c_l = train_step(train_x[idx], train_y[idx])
        pbar.set_postfix({'loss': f"{t_l:.3f}", 'rec': f"{r_l:.3f}", 'cl': f"{c_l:.3f}"})
        pbar.update(1)
    pbar.close()

    # Evaluation
    hits, mrr = 0, 0.0
    val_limit = 2000
    preds_eval, _ = hybrid_transformer.predict(test_x[:val_limit], batch_size=batch_size, verbose=0)

    for i in range(len(preds_eval)):
        top_20 = np.argsort(preds_eval[i])[-20:]
        if test_y[i] in top_20:
            hits += 1
            rank = np.where(top_20[::-1] == test_y[i])[0][0] + 1
            mrr += 1.0 / rank

    print(f"✨ Results Epoch {epoch+1} -> Recall@20: {(hits/val_limit)*100:.2f}% | MRR@20: {(mrr/val_limit)*100:.2f}%")
    K.clear_session(); gc.collect()

🔄 Loading data for Transformer Experiment...

🚀 TRANSFORMER HYBRID (C1+C2) | Vocab: 17377


Epoch 1/10: 100%|██████████| 650/650 [00:37<00:00, 17.35it/s, loss=5.696, rec=5.667, cl=0.290]


✨ Results Epoch 1 -> Recall@20: 59.25% | MRR@20: 22.29%


Epoch 2/10: 100%|██████████| 650/650 [00:34<00:00, 18.71it/s, loss=5.366, rec=5.340, cl=0.255]


✨ Results Epoch 2 -> Recall@20: 63.20% | MRR@20: 23.74%


...

✨ Results Epoch 8 -> Recall@20: 65.30% | MRR@20: 25.31%


Epoch 9/10: 100%|██████████| 650/650 [00:34<00:00, 18.71it/s, loss=4.121, rec=4.110, cl=0.114]


✨ Results Epoch 9 -> Recall@20: 64.70% | MRR@20: 25.92%


Epoch 10/10: 100%|██████████| 650/650 [00:34<00:00, 18.66it/s, loss=4.087, rec=4.074, cl=0.138]


✨ Results Epoch 10 -> Recall@20: 66.25% | MRR@20: 26.57%


In [ ]:
!pip install keras-self-attention

  Preparing metadata (setup.py) ... done
  Created wheel for keras-self-attention: filename=keras_self_attention-0.51.0-py3-none-any.whl size=18895 sha256=cacce8b86e47ce1c76f5b13e43c76e24598939b97d6cf6552db8c7efd3acf5eb
  Stored in directory: /root/.cache/pip/wheels/9a/9d/6e/09a0f61c2edeaea9f96fecdc67f31455c363bb44a4ddabe746
Successfully built keras-self-attention


In [ ]:
import numpy as np
import _pickle as cPickle
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, Lambda, Embedding
from tensorflow.keras.models import Model
from tensorflow.keras import backend as K
from keras_self_attention import SeqSelfAttention
from tqdm import tqdm
import gc

# ==========================================
# 2. DATA LOADING & PREP
# ==========================================
print("🔄 Loading data...")
with open('Datasets/yoochoose1_64/train.txt', 'rb') as f:
    train = cPickle.load(f)
with open('Datasets/yoochoose1_64/test.txt', 'rb') as f:
    test = cPickle.load(f)

vocab = np.load("Datasets/yoochoose1_64/vocab_yoochoose_64.npy")
total_vocab = len(vocab) + 1
convert_dict = {str(each): i + 1 for i, each in enumerate(vocab)}
convert_dict["padding_id"] = 0

def process_data(data_seq, labels_raw):
    x_out, y_out = [], []
    for seq, label in zip(data_seq, labels_raw):
        processed = [convert_dict.get(str(i), 0) for i in seq]
        if len(processed) > 10: processed = processed[:10]
        else: processed = processed + [0] * (10 - len(processed))
        x_out.append(processed)
        y_out.append(convert_dict.get(str(label), 0))
    return np.array(x_out), np.array(y_out)

train_x, train_y = process_data(train[0], train[1])
test_x, test_y = process_data(test[0], test[1])

# ==========================================
# 3. INITIALIZE LEARNABLE LOSS PARAMETERS
# ==========================================
log_var_rec = tf.Variable(0.0, trainable=True, name="log_var_rec", dtype=tf.float32)
log_var_cl = tf.Variable(0.0, trainable=True, name="log_var_cl", dtype=tf.float32)

# ==========================================
# 4. ARCHITECTURE (LSTM + ATTENTION)
# ==========================================
main_input = Input(shape=(10,))
emb = Embedding(total_vocab, 200, mask_zero=False)(main_input)
lstm_out = LSTM(100, return_sequences=True)(emb)
drop_out = Dropout(0.2)(lstm_out)
attn = SeqSelfAttention(attention_type=SeqSelfAttention.ATTENTION_TYPE_MUL)(drop_out)

z = Lambda(lambda xin: K.mean(xin, axis=1), name="latent")(attn)
output = Dense(total_vocab, activation='softmax')(z)

adaptive_model = Model(main_input, [output, z])
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

# ==========================================
# 5. ADAPTIVE TRAIN STEP
# ==========================================
@tf.function
def adaptive_train_step(xb, yb):
    with tf.GradientTape() as tape:
        preds, z_i = adaptive_model(xb, training=True)
        _, z_j = adaptive_model(xb, training=True)

        rec_loss = tf.reduce_mean(tf.keras.losses.sparse_categorical_crossentropy(yb, preds))

        z_i_n = tf.math.l2_normalize(z_i, axis=1)
        z_j_n = tf.math.l2_normalize(z_j, axis=1)
        logits = tf.matmul(z_i_n, z_j_n, transpose_b=True) / 0.07
        cl_loss = tf.reduce_mean(tf.nn.sparse_softmax_cross_entropy_with_logits(
            labels=tf.range(tf.shape(xb)[0]), logits=logits))

        # Uncertainty Weighting Formula
        precision_rec = tf.exp(-log_var_rec)
        precision_cl = tf.exp(-log_var_cl)

        weighted_rec = precision_rec * rec_loss + log_var_rec
        weighted_cl = precision_cl * cl_loss + log_var_cl
        total_loss = weighted_rec + weighted_cl

    trainable_vars = adaptive_model.trainable_variables + [log_var_rec, log_var_cl]
    grads = tape.gradient(total_loss, trainable_vars)
    optimizer.apply_gradients(zip(grads, trainable_vars))

    return total_loss, rec_loss, cl_loss, precision_rec, precision_cl

# ==========================================
# 6. EXECUTION LOOP
# ==========================================
print(f"🚀 RUNNING NOVELTY 3: LSTM + Task-Adaptive Weighting")

for epoch in range(5):
    indices = np.arange(len(train_x))
    np.random.shuffle(indices)

    pbar = tqdm(total=650, desc=f"Epoch {epoch+1}")
    for s in range(650):
        idx = indices[s*256 : (s+1)*256]
        if len(idx) < 256: continue

        t_l, r_l, c_l, p_rec, p_cl = adaptive_train_step(train_x[idx], train_y[idx])

        pbar.set_postfix({
            'L_rec': f"{r_l:.2f}",
            'W_rec': f"{p_rec:.2f}",
            'W_cl': f"{p_cl:.2f}"
        })
        pbar.update(1)
    pbar.close()

    # Evaluation
    hits, mrr = 0, 0.0
    val_limit = 2000
    preds_eval, _ = adaptive_model.predict(test_x[:val_limit], batch_size=256, verbose=0)
    for i in range(len(preds_eval)):
        top_20 = np.argsort(preds_eval[i])[-20:]
        if test_y[i] in top_20:
            hits += 1
            rank = np.where(top_20[::-1] == test_y[i])[0][0] + 1
            mrr += 1.0 / rank

    print(f"📊 Results Epoch {epoch+1} -> Recall@20: {(hits/val_limit)*100:.2f}% | MRR@20: {(mrr/val_limit)*100:.2f}%")
    K.clear_session(); gc.collect()

🔄 Loading data...
🚀 RUNNING NOVELTY 3: LSTM + Task-Adaptive Weighting



Epoch 1:   0%|          | 0/650 [00:15<?, ?it/s]

Epoch 1: 100%|██████████| 650/650 [00:17<00:00, 37.71it/s, L_rec=9.31, W_rec=0.58, W_cl=1.87]


📊 Results Epoch 1 -> Recall@20: 20.35% | MRR@20: 6.97%


Epoch 2: 100%|██████████| 650/650 [00:09<00:00, 68.06it/s, L_rec=5.51, W_rec=0.42, W_cl=3.41]


📊 Results Epoch 2 -> Recall@20: 57.00% | MRR@20: 21.95%


...

📊 Results Epoch 3 -> Recall@20: 59.65% | MRR@20: 23.05%


Epoch 4: 100%|██████████| 650/650 [00:09<00:00, 69.26it/s, L_rec=4.98, W_rec=0.29, W_cl=9.88]


📊 Results Epoch 4 -> Recall@20: 60.75% | MRR@20: 24.31%


Epoch 5: 100%|██████████| 650/650 [00:09<00:00, 68.35it/s, L_rec=4.88, W_rec=0.25, W_cl=14.15]


📊 Results Epoch 5 -> Recall@20: 61.50% | MRR@20: 24.38%
